In [11]:
import pandas as pd
flanking_seq_df = pd.read_csv("../rice_data/RiceDiversity.44K.MSU6.SNP_flanking_seq.txt", sep="\t")
msu7_map = pd.read_csv("../rice_data/RiceDiversity.44K.MSU6.SNP_Information.MSU7.txt", sep="\t")
sample_phylo = pd.read_csv("../rice_data/RiceDiversity_44K_Phenotypes_34traits_PLINK.txt", sep="\t")

In [5]:
from pyfaidx import Fasta
import pysam
fasta_path = "../rice_data/GCA_rice.fasta"
vcf_path = "../rice_data/RiceDiversity_44K_Genotypes_PLINK/sativas413.vcf"
ref_fasta = Fasta(fasta_path)
vcf = pysam.VariantFile(vcf_path)

In [13]:
# ── Sanity check: reproduce flanking_seq from GCA_rice.fasta (IRGSP1/MSU7) ──
#
# Two issues prevented the earlier attempt from working:
#   1. The FASTA uses ENA chromosome names (ENA|AP014957|AP014957.1, etc.)
#      rather than plain integers.
#   2. flanking_seq `pos` values use IRGSP.v4 / MSU6 coordinates, while the
#      FASTA uses the IRGSP1 / MSU7 coordinate system. The per-SNP mapping
#      lives in the SNP_Information file — but that file has a header/data
#      column-count mismatch (7 header fields, 8 data fields), so pandas
#      silently shifts column labels. The real IRGSP1-MSU7 positions end up
#      in the column pandas labels as `position.MSU.v6`.
#
# Strategy:
#   • For SNPs present in msu7_map: use the (correctly read) IRGSP1-MSU7
#     coordinate directly.
#   • For the ~7 k SNPs absent from msu7_map: interpolate the coordinate
#     offset from the nearest mapped neighbour on the same chromosome.
#     (Offsets are piecewise-constant within assembly blocks.)

import pandas as pd
import numpy as np
from pyfaidx import Fasta

fasta_path = "../rice_data/GCA_rice.fasta"
ref = Fasta(fasta_path)

# Map chromosome number 1-12 → FASTA sequence name
chrom_name = {i + 1: name for i, name in enumerate(ref.keys())}

# Re-read SNP_Information with correct column names.
# The file header has 7 fields but each data row has 8 (CHR is an extra
# leading field), so we name all 8 explicitly.
msu7_correct = pd.read_csv(
    "../rice_data/RiceDiversity.44K.MSU6.SNP_Information.MSU7.txt",
    sep="\t", header=0,
    names=["CHR", "SNPID", "cM",
           "pos_IRGSP_v4", "pos_MSU6",
           "pos_IRGSP1_MSU7", "MAF", "callrates"],
)
msu7_correct["pos_IRGSP1_MSU7"] = pd.to_numeric(
    msu7_correct["pos_IRGSP1_MSU7"], errors="coerce"
)

# Join flanking_seq with the coordinate mapping on snp_id / SNPID
df = flanking_seq_df.merge(
    msu7_correct[["SNPID", "CHR", "pos_IRGSP1_MSU7"]],
    left_on="snp_id", right_on="SNPID", how="left",
)
df["offset"] = df["pos_IRGSP1_MSU7"] - df["pos"]   # NaN for unmapped SNPs

# Interpolate offset for unmapped SNPs from nearest mapped neighbour
interpolated = df["offset"].copy()
for chrom, grp in df.groupby("chr"):
    mapped_m   = grp["offset"].notna()
    unmapped_m = ~mapped_m
    if not mapped_m.any() or not unmapped_m.any():
        continue
    mapped_pos = grp.loc[mapped_m,   "pos"].values
    mapped_off = grp.loc[mapped_m,   "offset"].values
    all_pos    = grp.loc[unmapped_m, "pos"].values
    nearest    = np.abs(all_pos[:, None] - mapped_pos[None, :]).argmin(axis=1)
    interpolated.loc[grp.index[unmapped_m]] = mapped_off[nearest]

df["pos_query"] = (df["pos"] + interpolated).round().astype(int)

# Query FASTA and compare against stored flanking sequences
chrom_arr = df["chr"].values
pos_arr   = df["pos_query"].values
left_arr  = df["X5p_MSU6"].values
right_arr = df["X3p_MSU6"].values

match = np.empty(len(df), dtype=object)
for i in range(len(df)):
    c      = int(chrom_arr[i])
    p      = int(pos_arr[i])
    window = ref[chrom_name[c]][p - 1 - 16 : p - 1 + 17].seq.upper()
    if len(window) != 33:
        match[i] = None
    else:
        match[i] = (window[:16] == left_arr[i].upper() and
                    window[17:] == right_arr[i].upper())

df["both_match"] = match

total   = sum(x is not None for x in match)
matched = sum(x is True    for x in match)
print(f"SNPs tested  : {total:,}")
print(f"Both flanks match: {matched:,}  ({100 * matched / total:.2f}%)")
print(f"Mismatches   : {total - matched:,}")

was_mapped = df["SNPID"].notna().values
m_m = sum(match[i] is True for i in range(len(df)) if     was_mapped[i] and match[i] is not None)
t_m = sum(1               for i in range(len(df)) if     was_mapped[i] and match[i] is not None)
m_i = sum(match[i] is True for i in range(len(df)) if not was_mapped[i] and match[i] is not None)
t_i = sum(1               for i in range(len(df)) if not was_mapped[i] and match[i] is not None)
print(f"\nDirectly mapped SNPs      : {m_m}/{t_m}  ({100*m_m/t_m:.2f}%)")
print(f"Offset-interpolated SNPs  : {m_i}/{t_i}  ({100*m_i/t_i:.2f}%)")


SNPs tested  : 42,755
Both flanks match: 42,504  (99.41%)
Mismatches   : 251

Directly mapped SNPs      : 35612/35674  (99.83%)
Offset-interpolated SNPs  : 6892/7081  (97.33%)
